In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/8588_Praxis/beige_books_full_report_cleaned.csv'
beige_book_df = pd.read_csv(file_path)

print(beige_book_df.shape)
beige_book_df.head(2)

(189, 3)


,date,content,cleaned_content
0,2002-01-01,<html> <head> <title>FRB: Beige Book - Full re...,"FRB: Beige Book - Full report January 16, 2002..."
1,2002-03-01,<html> <head> <title>FRB: Beige Book - Full re...,"FRB: Beige Book - Full report March 6, 2002 Su..."


In [ ]:
# Install and Load FinBERT
# !pip install transformers --quiet
# !pip install tqdm --quiet

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from tqdm import tqdm

In [ ]:
# Load Model

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"FinBERT model loaded on device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


FinBERT model loaded on device: cpu


In [ ]:
# Chunked Sentiment Function (Safe + Memory-Efficient)
def sentiment_score_finbert_safe(text, chunk_size=512):
    try:
        tokens = tokenizer.tokenize(text)
        sentiment_scores = []

        for i in range(0, len(tokens), chunk_size):
            chunk = tokens[i:i + chunk_size]
            chunk_text = tokenizer.convert_tokens_to_string(chunk)
            inputs = tokenizer(chunk_text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                sentiment_scores.append(probs[0].cpu().numpy())

        if sentiment_scores:
            avg_probs = np.mean(sentiment_scores, axis=0)
            return {
                'positive': float(avg_probs[0]),
                'negative': float(avg_probs[1]),
                'neutral':  float(avg_probs[2])
            }
        else:
            return {'positive': 0.0, 'negative': 0.0, 'neutral': 1.0}
    except Exception as e:
        print(f"Error processing document: {e}")
        return {'positive': 0.0, 'negative': 0.0, 'neutral': 1.0}

def get_compound_score(sentiment_dict):
    return sentiment_dict['positive'] - sentiment_dict['negative']

In [ ]:
print("Running FinBERT sentiment scoring on Beige Books...")
tqdm.pandas()
beige_book_df['sentiment_dict'] = beige_book_df['cleaned_content'].progress_apply(sentiment_score_finbert_safe)
beige_book_df['sentiment'] = beige_book_df['sentiment_dict'].apply(get_compound_score)

Running FinBERT sentiment scoring on Beige Books...


100%|██████████| 189/189 [5:14:57<00:00, 99.99s/it]


In [ ]:
output_path = '/content/drive/MyDrive/8588_Praxis/beige_books_sentiment_scores.csv'
beige_book_df[['date', 'sentiment', 'sentiment_dict']].to_csv(output_path, index=False)

print(f"Sentiment results saved to:\n{output_path}")

Sentiment results saved to:
/content/drive/MyDrive/8588_Praxis/beige_books_sentiment_scores.csv


# Brainstorm Approach
Splitting the Beige Book text into 512-token chunks can impact sentiment accuracy, especially if it breaks sentences or paragraphs in the middle, which disrupts the context that models like FinBERT rely on.

Use nltk or spacy to split the document into sentences first, then aggregate sentences until you approach the 512-token limit


## Improving FinBERT Sentiment Accuracy with Sentence-Aware Chunking

When running FinBERT for sentiment analysis on long Beige Book documents, splitting the text into fixed 512-token chunks can degrade performance because:

- Sentences or paragraphs may get split mid-way
- Important contextual cues may be lost
- The model receives incomplete semantic units

Since FinBERT relies heavily on context, this can result in lower accuracy and unstable sentiment predictions.

---

### Why Use Sentence-Aware Chunking?

To address these issues, we use a smarter method:

1. **Split the document into individual sentences** using `nltk.sent_tokenize()` or `spaCy`.
2. **Aggregate full sentences into chunks** without exceeding the 512-token limit.
3. **Preserve sentence integrity**: each chunk ends cleanly at a sentence boundary.
4. **Run FinBERT inference on each chunk** independently.
5. **Average the softmax probabilities** across all chunks for a final sentiment distribution.

---

### Benefits

| Problem with Naive Token Chunking      | Solution via Sentence-Aware Chunking         |
|----------------------------------------|-----------------------------------------------|
| Sentences split mid-way                | Full sentences are preserved                  |
| Reduced context for sentiment analysis | Maintains semantic integrity                  |
| Lower model confidence and accuracy    | Produces more reliable predictions            |
| Risk of token limit overflow           | Avoids truncation and execution errors        |

---

### Summary

Sentence-aware chunking significantly improves the **stability**, **accuracy**, and **interpretability** of FinBERT sentiment scores, especially for large and nuanced documents like the Beige Book.

In [ ]:
# !pip install -U spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
def sentiment_score_finbert_spacy(text, chunk_token_limit=512):
    try:
        doc = nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents]

        sentiment_scores = []
        current_chunk = ""
        current_tokens = 0

        for sentence in sentences:
            token_count = len(tokenizer.tokenize(sentence))
            if current_tokens + token_count > chunk_token_limit:
                # Score current chunk
                inputs = tokenizer(current_chunk, return_tensors="pt", truncation=True, max_length=512)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                with torch.no_grad():
                    outputs = model(**inputs)
                    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                    sentiment_scores.append(probs[0].cpu().numpy())
                # Reset chunk
                current_chunk = sentence
                current_tokens = token_count
            else:
                current_chunk += " " + sentence
                current_tokens += token_count

        # Score last chunk
        if current_chunk:
            inputs = tokenizer(current_chunk, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                sentiment_scores.append(probs[0].cpu().numpy())

        # Aggregate
        if sentiment_scores:
            avg_probs = np.mean(sentiment_scores, axis=0)
            return {
                'positive_spacy': float(avg_probs[0]),
                'negative_spacy': float(avg_probs[1]),
                'neutral_spacy':  float(avg_probs[2])
            }
        else:
            return {'positive_spacy': 0.0, 'negative_spacy': 0.0, 'neutral_spacy': 1.0}
    except Exception as e:
        print(f"Error processing document: {e}")
        return {'positive_spacy': 0.0, 'negative_spacy': 0.0, 'neutral_spacy': 1.0}

In [ ]:
def get_compound_score_spacy(sentiment_dict):
    return sentiment_dict['positive_spacy'] - sentiment_dict['negative_spacy']

print("Running FinBERT (Spacy-Aware) sentiment scoring on Beige Books...")
tqdm.pandas()
beige_book_df['sentiment_dict_spacy'] = beige_book_df['cleaned_content'].progress_apply(sentiment_score_finbert_spacy)
beige_book_df['sentiment_spacy'] = beige_book_df['sentiment_dict_spacy'].apply(get_compound_score_spacy)

Running FinBERT (Spacy-Aware) sentiment scoring on Beige Books...


100%|██████████| 189/189 [4:22:59<00:00, 83.49s/it]


In [ ]:
output_path = '/content/drive/MyDrive/8588_Praxis/beige_books_sentiment_scores_spacy.csv'
beige_book_df[['date', 'sentiment_spacy', 'sentiment_dict_spacy']].to_csv(output_path, index=False)

print(f"Sentiment results saved to:\n{output_path}")

Sentiment results saved to:
/content/drive/MyDrive/8588_Praxis/beige_books_sentiment_scores_spacy.csv


In [ ]:
print("✅ Kernel still alive")

✅ Kernel still alive


In [ ]:
from nltk.tokenize import sent_tokenize

def sentiment_score_finbert_sentence_aware(text, chunk_size=512):
    try:
        # Split text into sentences
        sentences = sent_tokenize(text)
        current_chunk = ""
        sentiment_scores = []

        for sentence in sentences:
            # Check token length if adding this sentence exceeds chunk_size
            temp_chunk = current_chunk + " " + sentence if current_chunk else sentence
            token_count = len(tokenizer.tokenize(temp_chunk))

            if token_count <= chunk_size:
                current_chunk = temp_chunk
            else:
                # Process current chunk
                inputs = tokenizer(current_chunk, return_tensors="pt", truncation=True, max_length=512)
                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model(**inputs)
                    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                    sentiment_scores.append(probs[0].cpu().numpy())

                # Start new chunk with current sentence
                current_chunk = sentence

        # Process any remaining text
        if current_chunk:
            inputs = tokenizer(current_chunk, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                sentiment_scores.append(probs[0].cpu().numpy())

        # Aggregate sentiment scores
        if sentiment_scores:
            avg_probs = np.mean(sentiment_scores, axis=0)
            return {
                'positive': float(avg_probs[0]),
                'negative': float(avg_probs[1]),
                'neutral':  float(avg_probs[2])
            }
        else:
            return {'positive': 0.0, 'negative': 0.0, 'neutral': 1.0}

    except Exception as e:
        print(f"Error processing document: {e}")
        return {'positive': 0.0, 'negative': 0.0, 'neutral': 1.0}


def get_compound_score(sentiment_dict):
    return sentiment_dict['positive'] - sentiment_dict['negative']

In [ ]:
print("Running FinBERT sentiment scoring on Beige Books with sentence-aware chunking...")
tqdm.pandas()

# Apply sentence-aware sentiment scoring
beige_book_df['sentiment_dict_sentence_aware'] = beige_book_df['cleaned_content'].progress_apply(sentiment_score_finbert_sentence_aware)

# Compute compound score: positive - negative
beige_book_df['sentiment_sentence_aware'] = beige_book_df['sentiment_dict_sentence_aware'].apply(get_compound_score)

Running FinBERT sentiment scoring on Beige Books with sentence-aware chunking...


100%|██████████| 189/189 [00:00<00:00, 1937.67it/s]

Error processing document: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizers/punkt_tab/english/

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************

Error processing document: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizer

In [ ]:
# Export sentence-aware FinBERT sentiment results
output_path_sentence_aware = '/content/drive/MyDrive/8588_Praxis/beige_books_sentiment_scores_sentence_aware.csv'

beige_book_df[['date', 'sentiment_sentence_aware', 'sentiment_dict_sentence_aware']].to_csv(
    output_path_sentence_aware,
    index=False
)

print(f"Sentence-aware sentiment results saved to:\n{output_path_sentence_aware}")